In [ ]:
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
documents[10]

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm":
        documents_llm.append(doc)

len(documents_llm)

In [ ]:
documents = documents_llm

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json
user_prompt = json.dumps(doc)

In [ ]:
user_prompt

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
response.output_parsed.questions

In [ ]:
doc

In [ ]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
usage

In [ ]:
from evaluation_utils import calc_price

In [ ]:
calc_price(usage)

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(records)

In [ ]:
from evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
ground_truth[10]

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)